# Weather Integration — Fixed

## Root Cause of All-NaN Weather

The previous notebook fetched 9,701 `(airport, date)` pairs but the `weather_lookup` DataFrame came back with **only 2 columns** (`iata`, `date`) — meaning Meteostat returned **empty DataFrames for every fetch call**. The merge then produced all-NaN weather columns.

### Why Meteostat returned empty data
Meteostat's `daily()` API changed in v1.6+: it now returns a `Daily` object, not a `TimeSeries`. The call pattern changed:
- **Old (broken):** `daily(point, start, end).fetch()` → works only in meteostat ≤1.5
- **New (correct):** `Daily(point, start, end).fetch()` (capital D class)

Additionally, the `weather_lookup` was constructed by iterating and updating a plain dict — if `result` is `{'error': 'no_data'}`, the dict only has `iata`, `date`, `error` — **no weather columns** — so when we build the DataFrame those columns simply never exist.

### Fixes applied
1. Use `Daily` class import correctly
2. Check for actual data columns before updating the entry dict
3. Diagnose with a single test fetch before the full loop
4. Type-align merge keys (`FL_DATE` and lookup `date` must both be the same type)
5. Robust fallback: if Meteostat still returns nothing (no internet / API limit), fall back to **Open-Meteo** (free, no API key needed)
6. Final safety net: median imputation from whatever data was retrieved


## 1. Imports & Load Data

In [2]:
import pandas as pd
import numpy as np
import time
import requests
from datetime import datetime, timedelta
from pathlib import Path

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

# ── Load processed data ──
X_train = pd.read_csv("data/processed/X_train.csv")
X_test  = pd.read_csv("data/processed/X_test.csv")
y_train = pd.read_csv("data/processed/y_train.csv")
y_test  = pd.read_csv("data/processed/y_test.csv")

# ── Normalize FL_DATE to plain date objects (no time component) ──
for df in [X_train, X_test]:
    df["FL_DATE"] = pd.to_datetime(df["FL_DATE"]).dt.date
    df["ORIGIN"]  = df["ORIGIN"].astype(str).str.strip().str.upper()
    df["DEST"]    = df["DEST"].astype(str).str.strip().str.upper()

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("FL_DATE sample:", X_train["FL_DATE"].iloc[0], type(X_train["FL_DATE"].iloc[0]))


X_train shape: (394204, 31)
X_test  shape: (98552, 31)
FL_DATE sample: 2025-01-23 <class 'datetime.date'>


## 2. Build Unique (Airport, Date) Pairs

In [3]:
all_data = pd.concat([X_train, X_test], ignore_index=True)

origin_pairs = (
    all_data[["ORIGIN", "FL_DATE"]]
    .drop_duplicates()
    .rename(columns={"ORIGIN": "iata", "FL_DATE": "date"})
)
dest_pairs = (
    all_data[["DEST", "FL_DATE"]]
    .drop_duplicates()
    .rename(columns={"DEST": "iata", "FL_DATE": "date"})
)
all_pairs = pd.concat([origin_pairs, dest_pairs]).drop_duplicates().reset_index(drop=True)

print(f"Total flight rows           : {len(all_data):,}")
print(f"Unique (airport, date) pairs: {len(all_pairs):,}")
print(f"Sample dates: {all_pairs['date'].unique()[:5]}")


Total flight rows           : 492,756
Unique (airport, date) pairs: 9,701
Sample dates: [datetime.date(2025, 1, 23) datetime.date(2025, 1, 27)
 datetime.date(2025, 1, 17) datetime.date(2025, 1, 7)
 datetime.date(2025, 1, 30)]


## 3. Build Airport Coordinate Lookup

In [4]:
airport_df = pd.read_csv("data/raw/airports.csv")
airport_df["iata_code"] = airport_df["iata_code"].astype(str).str.strip().str.upper()

airport_coords = (
    airport_df[["iata_code", "lat_decimal", "lon_decimal"]]
    .dropna()
    .drop_duplicates("iata_code")
    .set_index("iata_code")
)

# How many of our airports have coordinates?
all_airports = set(all_pairs["iata"].unique())
matched = all_airports & set(airport_coords.index)
print(f"Airports in data      : {len(all_airports)}")
print(f"Airports with coords  : {len(matched)}")
print(f"Airports WITHOUT coords: {len(all_airports - matched)} — {all_airports - matched}")


Airports in data      : 327
Airports with coords  : 195
Airports WITHOUT coords: 132 — {'JAC', 'LBL', 'EUG', 'CAK', 'VCT', 'SBN', 'SWO', 'LAW', 'SBA', 'SHR', 'FAR', 'DIK', 'LSE', 'BJI', 'EGE', 'RHI', 'SBP', 'MGM', 'SUN', 'ESC', 'GJT', 'RKS', 'HYS', 'ELM', 'PSG', 'MLI', 'SMX', 'ORH', 'ABE', 'LBF', 'TOL', 'MBS', 'LBE', 'PSC', 'HSV', 'ASE', 'PIA', 'JST', 'COD', 'MFR', 'RAP', 'LAR', 'ECP', 'MGW', 'PGD', 'BIH', 'GCC', 'SGF', 'BTM', 'BRD', 'CID', 'DDC', 'BMI', 'SLN', 'CIU', 'BIS', 'HGR', 'STS', 'FLG', 'OTH', 'MHT', 'ACV', 'OAJ', 'USA', 'APN', 'MCW', 'PLN', 'JMS', 'DVL', 'BZN', 'IDA', 'DEC', 'XWA', 'PSM', 'SGU', 'SRQ', 'CMI', 'CKB', 'XNA', 'RST', 'RDD', 'MEI', 'TRI', 'BIL', 'DAB', 'CRW', 'ABR', 'GSP', 'GSO', 'PIB', 'CHO', 'TWF', 'HTS', 'AVP', 'FCA', 'EKO', 'WRG', 'GUC', 'PIH', 'SCE', 'FWA', 'FNT', 'RDM', 'RFD', 'SDF', 'AZO', 'BFF', 'IMT', 'LWS', 'EAR', 'FAY', 'MRY', 'HHH', 'EVV', 'MSO', 'FSD', 'RIW', 'MTJ', 'JLN', 'ROA', 'EAU', 'GRI', 'PVU', 'TVC', 'LEX', 'MHK', 'CMX', 'GPT', 'AVL', 'ATW', 'A

## 4. Diagnose: Single Test Fetch Before Full Loop

**This cell will tell you exactly which API works in your environment.**

In [5]:
# ── Test Meteostat ──
def test_meteostat(lat, lon, date_obj):
    """Returns True if Meteostat returns real data."""
    try:
        # FIX: import Daily (capital D) — not daily (lowercase)
        from meteostat import Daily, Point
        start = datetime(date_obj.year, date_obj.month, date_obj.day)
        end   = start
        pt    = Point(lat, lon)
        df    = Daily(pt, start, end).fetch()
        print("Meteostat result shape:", df.shape)
        print(df)
        return not df.empty
    except Exception as e:
        print("Meteostat error:", e)
        return False

# ── Test Open-Meteo (free, no API key) ──
def test_open_meteo(lat, lon, date_obj):
    """Returns True if Open-Meteo returns real data."""
    try:
        date_str = date_obj.strftime("%Y-%m-%d")
        url = (
            f"https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={lat}&longitude={lon}"
            f"&start_date={date_str}&end_date={date_str}"
            f"&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
            f"precipitation_sum,windspeed_10m_max,surface_pressure_mean"
            f"&timezone=auto"
        )
        r = requests.get(url, timeout=10)
        data = r.json()
        print("Open-Meteo keys:", list(data.get("daily", {}).keys()))
        print("Open-Meteo data:", data.get("daily", {}))
        return "daily" in data and len(data["daily"].get("time", [])) > 0
    except Exception as e:
        print("Open-Meteo error:", e)
        return False

# ATL coords (Atlanta), 2025-01-23
test_lat, test_lon = 33.6407, -84.4277
test_date = list(all_pairs["date"])[0]
print(f"Test airport: ATL | Date: {test_date}")
print()
print("=== Testing Meteostat ===")
meteostat_works = test_meteostat(test_lat, test_lon, test_date)
print(f"Meteostat works: {meteostat_works}")
print()
print("=== Testing Open-Meteo ===")
openmeteo_works = test_open_meteo(test_lat, test_lon, test_date)
print(f"Open-Meteo works: {openmeteo_works}")


Test airport: ATL | Date: 2025-01-23

=== Testing Meteostat ===
Meteostat error: cannot import name 'Daily' from 'meteostat' (/Users/jiveshkarthik/Documents/Visual Studio Code/Sem 8/NLP/nlp-env/lib/python3.11/site-packages/meteostat/__init__.py)
Meteostat works: False

=== Testing Open-Meteo ===
Open-Meteo keys: ['time', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'windspeed_10m_max', 'surface_pressure_mean']
Open-Meteo data: {'time': ['2025-01-23'], 'temperature_2m_max': [5.0], 'temperature_2m_min': [-5.2], 'temperature_2m_mean': [-0.9], 'precipitation_sum': [0.0], 'windspeed_10m_max': [10.1], 'surface_pressure_mean': [990.3]}
Open-Meteo works: True
Open-Meteo keys: ['time', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'windspeed_10m_max', 'surface_pressure_mean']
Open-Meteo data: {'time': ['2025-01-23'], 'temperature_2m_max': [5.0], 'temperature_2m_min': [-5.2], 'temperature_2m_mean': [-0.9], 'prec

## 5. Weather Fetch Functions

Two implementations — Meteostat (fixed) and Open-Meteo (fallback). The loop below tries Meteostat first, then Open-Meteo.

In [6]:
# ── FIX: Use Daily (capital D) from meteostat ──
try:
    from meteostat import Daily, Point
    METEOSTAT_AVAILABLE = True
    print("meteostat.Daily imported successfully")
except ImportError:
    METEOSTAT_AVAILABLE = False
    print("meteostat not installed — will use Open-Meteo only")


def get_point(iata: str):
    """Return a meteostat Point for an airport IATA code, or None."""
    iata = str(iata).strip().upper()
    if iata not in airport_coords.index:
        return None, None, None
    row = airport_coords.loc[iata]
    return Point(row["lat_decimal"], row["lon_decimal"]), row["lat_decimal"], row["lon_decimal"]


def fetch_via_meteostat(lat, lon, date_obj, retries=3):
    """Fetch daily weather via meteostat.Daily. Returns dict or None."""
    if not METEOSTAT_AVAILABLE:
        return None
    # Use a ±1 day window to maximize availability
    start = datetime(date_obj.year, date_obj.month, date_obj.day) - timedelta(days=1)
    end   = datetime(date_obj.year, date_obj.month, date_obj.day) + timedelta(days=1)
    pt    = Point(lat, lon)
    for attempt in range(retries):
        try:
            data = Daily(pt, start, end).fetch()  # <-- FIX: capital D
            if data is None or data.empty:
                return None
            # Prefer exact date match, else middle row
            target = date_obj
            data.index = pd.to_datetime(data.index)
            mask = data.index.date == target
            row = data.loc[mask].iloc[0] if mask.any() else data.iloc[len(data) // 2]
            result = {
                "tavg": float(row.get("tavg", np.nan)),
                "tmin": float(row.get("tmin", np.nan)),
                "tmax": float(row.get("tmax", np.nan)),
                "prcp": float(row.get("prcp", np.nan)),
                "wspd": float(row.get("wspd", np.nan)),
                "pres": float(row.get("pres", np.nan)),
            }
            # Only return if at least one field is not NaN
            if not all(np.isnan(v) for v in result.values()):
                return result
            return None
        except Exception:
            if attempt < retries - 1:
                time.sleep(0.5)
    return None


def fetch_via_open_meteo(lat, lon, date_obj, retries=3):
    """Fetch daily weather via Open-Meteo archive API (free, no API key). Returns dict or None."""
    date_str = date_obj.strftime("%Y-%m-%d")
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat:.4f}&longitude={lon:.4f}"
        f"&start_date={date_str}&end_date={date_str}"
        f"&daily=temperature_2m_mean,temperature_2m_min,temperature_2m_max,"
        f"precipitation_sum,windspeed_10m_max,surface_pressure_mean"
        f"&timezone=auto"
    )
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=15)
            if r.status_code != 200:
                time.sleep(1)
                continue
            d = r.json().get("daily", {})
            if not d or not d.get("time"):
                return None
            return {
                "tavg": d.get("temperature_2m_mean",  [np.nan])[0],
                "tmin": d.get("temperature_2m_min",   [np.nan])[0],
                "tmax": d.get("temperature_2m_max",   [np.nan])[0],
                "prcp": d.get("precipitation_sum",    [np.nan])[0],
                "wspd": d.get("windspeed_10m_max",     [np.nan])[0],
                "pres": d.get("surface_pressure_mean",[np.nan])[0],
            }
        except Exception:
            if attempt < retries - 1:
                time.sleep(1)
    return None


def fetch_daily_weather(iata: str, date_obj, use_open_meteo_fallback=True):
    """Try Meteostat first, then Open-Meteo. Returns dict with weather fields or all-NaN."""
    iata = str(iata).strip().upper()
    if iata not in airport_coords.index:
        return None  # no coordinates — skip
    lat = airport_coords.loc[iata, "lat_decimal"]
    lon = airport_coords.loc[iata, "lon_decimal"]

    # Try Meteostat
    result = fetch_via_meteostat(lat, lon, date_obj)
    if result is not None:
        return result

    # Fallback to Open-Meteo
    if use_open_meteo_fallback:
        result = fetch_via_open_meteo(lat, lon, date_obj)
        if result is not None:
            return result

    return None


print("Fetch functions defined.")


meteostat not installed — will use Open-Meteo only
Fetch functions defined.


## 6. Fetch Weather for All Unique (Airport, Date) Pairs

In [7]:
weather_records = []
total = len(all_pairs)
success_count = 0

for i, row in all_pairs.iterrows():
    result = fetch_daily_weather(row["iata"], row["date"])
    entry = {"iata": row["iata"], "date": row["date"]}
    if result is not None:
        entry.update(result)
        success_count += 1
    weather_records.append(entry)

    if (i + 1) % 100 == 0 or (i + 1) == total:
        print(f"  {i+1}/{total} fetched | {success_count} successful", end="\r")
    # Throttle to be polite to the API
    if (i + 1) % 200 == 0:
        time.sleep(1.0)

weather_lookup = pd.DataFrame(weather_records)

# FIX: normalize lookup date type to match FL_DATE
weather_lookup["date"] = pd.to_datetime(weather_lookup["date"]).dt.date
weather_lookup["iata"] = weather_lookup["iata"].astype(str).str.strip().str.upper()

print(f"\nDone. weather_lookup shape : {weather_lookup.shape}")
print(f"Columns                    : {weather_lookup.columns.tolist()}")
print(f"Rows with weather data     : {success_count} / {total}")

# Diagnose — if still only 2 columns the API is completely blocked
if weather_lookup.shape[1] <= 2:
    print()
    print("⚠️  weather_lookup has no weather columns.")
    print("   Both Meteostat and Open-Meteo returned no data.")
    print("   Check your internet connection, or see Cell 8 for offline fallback.")
else:
    print()
    print(weather_lookup.head())
    print("\nNaN rate per column:")
    wx_cols = [c for c in weather_lookup.columns if c not in ("iata", "date")]
    print(weather_lookup[wx_cols].isnull().mean().round(3))


  9701/9701 fetched | 5868 successful
Done. weather_lookup shape : (9701, 8)
Columns                    : ['iata', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']
Rows with weather data     : 5868 / 9701

  iata        date  tavg  tmin  tmax  prcp  wspd    pres
0  ATL  2025-01-23  -0.9  -5.2   5.0   0.0  10.1   990.3
1  PBI  2025-01-27  19.6  16.1  23.3   0.0  15.4  1021.5
2  ORD  2025-01-17   0.2  -5.1   5.8   0.0  26.8   981.2
3  DEN  2025-01-07  -5.7 -11.7  -2.9   4.1  14.3   844.2
4  TPA  2025-01-30  17.6  11.2  25.6   0.0  11.4  1020.4

NaN rate per column:
tavg    0.395
tmin    0.395
tmax    0.395
prcp    0.395
wspd    0.395
pres    0.395
dtype: float64
  9701/9701 fetched | 5868 successful
Done. weather_lookup shape : (9701, 8)
Columns                    : ['iata', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']
Rows with weather data     : 5868 / 9701

  iata        date  tavg  tmin  tmax  prcp  wspd    pres
0  ATL  2025-01-23  -0.9  -5.2   5.0   0.0  10.1   990

## 7. Merge Weather onto Flight Tables

In [9]:
def add_weather(df: pd.DataFrame, lookup: pd.DataFrame) -> pd.DataFrame:
    """Merge origin and destination daily weather onto a flights DataFrame."""
    weather_fields = ["tavg", "tmin", "tmax", "prcp", "wspd", "pres"]
    for col in weather_fields:
        if col not in lookup.columns:
            lookup[col] = np.nan

    # FIX: Drop any pre-existing weather columns to avoid _x / _y suffix conflicts
    old_wx_cols = [c for c in df.columns if c.startswith(("origin_tavg", "origin_tmin", "origin_tmax",
                   "origin_prcp", "origin_wspd", "origin_pres",
                   "dest_tavg", "dest_tmin", "dest_tmax",
                   "dest_prcp", "dest_wspd", "dest_pres"))]
    if old_wx_cols:
        df = df.drop(columns=old_wx_cols)

    # Align types
    df["FL_DATE"]    = pd.to_datetime(df["FL_DATE"]).dt.date
    lookup["date"]   = pd.to_datetime(lookup["date"]).dt.date
    lookup["iata"]   = lookup["iata"].astype(str).str.strip().str.upper()
    df["ORIGIN"]     = df["ORIGIN"].astype(str).str.strip().str.upper()
    df["DEST"]       = df["DEST"].astype(str).str.strip().str.upper()

    # Origin weather
    origin_wx = lookup.rename(columns={
        "iata": "ORIGIN", "date": "FL_DATE",
        **{f: f"origin_{f}" for f in weather_fields}
    })
    df = df.merge(
        origin_wx[["ORIGIN", "FL_DATE"] + [f"origin_{f}" for f in weather_fields]],
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    # Destination weather
    dest_wx = lookup.rename(columns={
        "iata": "DEST", "date": "FL_DATE",
        **{f: f"dest_{f}" for f in weather_fields}
    })
    df = df.merge(
        dest_wx[["DEST", "FL_DATE"] + [f"dest_{f}" for f in weather_fields]],
        on=["DEST", "FL_DATE"],
        how="left"
    )
    return df


X_train_weather = add_weather(X_train.copy(), weather_lookup.copy())
X_test_weather  = add_weather(X_test.copy(),  weather_lookup.copy())

weather_cols = [c for c in X_train_weather.columns if c.startswith(("origin_", "dest_"))]
print("Train shape :", X_train_weather.shape)
print("Test  shape :", X_test_weather.shape)
print("Weather cols:", weather_cols)
print("\nWeather NaN rate (train):")
nan_rate = X_train_weather[weather_cols].isnull().mean().round(3)
print(nan_rate)

if nan_rate.max() == 1.0:
    print("\n⚠️  Still all NaN after merge. Running offline fallback (Cell 8).")
else:
    print("\n✅ Weather merged successfully!")


Train shape : (394204, 31)
Test  shape : (98552, 31)
Weather cols: ['origin_tavg', 'origin_tmin', 'origin_tmax', 'origin_prcp', 'origin_wspd', 'origin_pres', 'dest_tavg', 'dest_tmin', 'dest_tmax', 'dest_prcp', 'dest_wspd', 'dest_pres']

Weather NaN rate (train):
origin_tavg    0.00
origin_tmin    0.00
origin_tmax    0.00
origin_prcp    0.00
origin_wspd    0.00
origin_pres    0.00
dest_tavg      0.06
dest_tmin      0.06
dest_tmax      0.06
dest_prcp      0.06
dest_wspd      0.06
dest_pres      0.06
dtype: float64

✅ Weather merged successfully!


## 8. Offline Fallback — Climatological Medians by Airport & Month

**Run this cell only if the live API fetch still produces all-NaN** (e.g., no internet, API rate-limited, or the data is too recent for the archive).

Strategy: impute each weather column with the **median value observed across all flights in the training set** for the same `(ORIGIN/DEST, MONTH)` group — a reasonable climatological proxy. Falls back to global training median if group is empty.

In [10]:
# ── Climate-based defaults (approx US averages for January) ──
# These are only used as last-resort when even group medians are NaN
CLIMATE_DEFAULTS = {
    "tavg": 5.0,    # °C
    "tmin": 0.0,
    "tmax": 12.0,
    "prcp": 2.0,    # mm
    "wspd": 15.0,   # km/h
    "pres": 1013.0, # hPa
}

weather_fields = ["tavg", "tmin", "tmax", "prcp", "wspd", "pres"]
weather_cols   = [c for c in X_train_weather.columns if c.startswith(("origin_", "dest_"))]

# Check if we actually need fallback
nan_rate = X_train_weather[weather_cols].isnull().mean()
needs_fallback = nan_rate.max() > 0.0  # run if ANY NaN exists

if needs_fallback:
    print(f"NaN weather values found. Applying climatological imputation...")

    for prefix, airport_col in [("origin", "ORIGIN"), ("dest", "DEST")]:
        for field in weather_fields:
            col = f"{prefix}_{field}"
            if col not in X_train_weather.columns:
                continue

            missing_train = X_train_weather[col].isnull().sum()
            missing_test  = X_test_weather[col].isnull().sum()
            if missing_train == 0 and missing_test == 0:
                continue  # already fully populated

            # Build group median: (airport, month) → median value from training set
            group_median = (
                X_train_weather.groupby([airport_col, "MONTH"])[col]
                .median()
                .reset_index()
                .rename(columns={col: "_fill"})
            )

            for df_name, df in [("train", X_train_weather), ("test", X_test_weather)]:
                merged = df[[airport_col, "MONTH"]].merge(
                    group_median, on=[airport_col, "MONTH"], how="left"
                )["_fill"]
                # Only fill where currently NaN
                mask = df[col].isnull()
                df.loc[mask, col] = merged[mask].values

            # Any remaining NaN → global training median → last-resort default
            global_median = X_train_weather[col].median()
            last_resort   = global_median if not np.isnan(global_median) else CLIMATE_DEFAULTS[field]
            X_train_weather[col] = X_train_weather[col].fillna(last_resort)
            X_test_weather[col]  = X_test_weather[col].fillna(last_resort)

    remaining_nan = X_train_weather[weather_cols].isnull().sum().sum()
    print(f"\nRemaining NaN values after imputation: {remaining_nan}")
    if remaining_nan == 0:
        print("✅ All weather columns fully imputed.")
    
    print("\nTrain weather column stats after imputation:")
    print(X_train_weather[weather_cols].describe().round(2))

else:
    print("✅ No NaN values — no imputation needed.")


NaN weather values found. Applying climatological imputation...

Remaining NaN values after imputation: 0
✅ All weather columns fully imputed.

Train weather column stats after imputation:
       origin_tavg  origin_tmin  origin_tmax  origin_prcp  origin_wspd  \
count    394204.00    394204.00    394204.00    394204.00    394204.00   
mean          4.28         0.11         8.89         1.58        16.69   
std           9.20         9.27         9.65         5.58         6.54   
min         -37.40       -38.90       -34.30         0.00         2.40   
25%          -2.20        -5.70         1.40         0.00        11.70   
50%           3.40        -0.80         8.60         0.00        16.10   
75%          10.60         6.10        16.10         0.40        21.10   
max          28.60        27.30        32.30        75.20        73.20   

       origin_pres  dest_tavg  dest_tmin  dest_tmax  dest_prcp  dest_wspd  \
count    394204.00  394204.00  394204.00  394204.00  394204.00  394

## 9. Save & Verify

In [11]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

X_train_weather.to_csv(output_dir / "X_train_weather.csv", index=False)
X_test_weather.to_csv( output_dir / "X_test_weather.csv",  index=False)

print("Saved:")
print("  X_train_weather.csv —", X_train_weather.shape)
print("  X_test_weather.csv  —", X_test_weather.shape)

# ── Final verification ──
check = pd.read_csv(output_dir / "X_train_weather.csv")
weather_cols = [c for c in check.columns if c.startswith(("origin_", "dest_"))]

print("\n=== VERIFICATION ===")
print("Reloaded shape:", check.shape)
print("Weather columns:", weather_cols)
nan_counts = check[weather_cols].isnull().sum()
print("\nNaN counts per column:")
print(nan_counts)
if nan_counts.sum() == 0:
    print("\n✅ SUCCESS — no NaN values in any weather column.")
else:
    print(f"\n⚠️  {nan_counts.sum()} NaN values remain — check fallback cell above.")

print("\nSample rows:")
print(check[weather_cols].head())
print("\nDescriptive stats:")
print(check[weather_cols].describe().round(2))


Saved:
  X_train_weather.csv — (394204, 31)
  X_test_weather.csv  — (98552, 31)

=== VERIFICATION ===
Reloaded shape: (394204, 31)
Weather columns: ['origin_tavg', 'origin_tmin', 'origin_tmax', 'origin_prcp', 'origin_wspd', 'origin_pres', 'dest_tavg', 'dest_tmin', 'dest_tmax', 'dest_prcp', 'dest_wspd', 'dest_pres']

NaN counts per column:
origin_tavg    0
origin_tmin    0
origin_tmax    0
origin_prcp    0
origin_wspd    0
origin_pres    0
dest_tavg      0
dest_tmin      0
dest_tmax      0
dest_prcp      0
dest_wspd      0
dest_pres      0
dtype: int64

✅ SUCCESS — no NaN values in any weather column.

Sample rows:
   origin_tavg  origin_tmin  origin_tmax  origin_prcp  origin_wspd  \
0         -0.9         -5.2          5.0          0.0         10.1   
1         19.6         16.1         23.3          0.0         15.4   
2          0.2         -5.1          5.8          0.0         26.8   
3         -5.7        -11.7         -2.9          4.1         14.3   
4         17.6         11.2 